# 레이블 없는 데이터를 활용한 사전 훈련 목차
* [Chapter 1 텍스트 생성 모델 평가하기](#chapter1)

## Chapter 1 텍스트 생성 모델 평가하기 <a class="anchor" id="chapter1"></a>
1. 텍스트 생성 과정을 간략히 정리한 후 텍스트 생성을 위해 LLM을 준비한다.

2. 생성된 텍스트의 품질을 평가하는 기본적인 방법을 소개한다.

    ![구현 순서](image/05-01-process2.png)

3. GPT를 사용해 텍스트 생성하기
   - LLM을 준비하고 텍스트 생성 과정을 간단하게 복습한다.

      ![구현 순서2](image/05-01-processTxt2.png)

In [1]:
# 깃허브에서 previous_chapters.py 파일을 다운로드합니다.
!wget https://bit.ly/3HlFmc8 -O previous_chapters.py

--2025-10-16 10:54:22--  https://bit.ly/3HlFmc8
Resolving bit.ly (bit.ly)... 67.199.248.10, 67.199.248.11
Connecting to bit.ly (bit.ly)|67.199.248.10|:443... connected.
HTTP request sent, awaiting response... 301 Moved Permanently
Location: https://raw.githubusercontent.com/rickiepark/llm-from-scratch/refs/heads/main/ch05/01_main-chapter-code/previous_chapters.py [following]
--2025-10-16 10:54:23--  https://raw.githubusercontent.com/rickiepark/llm-from-scratch/refs/heads/main/ch05/01_main-chapter-code/previous_chapters.py
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.111.133, 185.199.108.133, 185.199.109.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.111.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 9905 (9.7K) [text/plain]
Saving to: ‘previous_chapters.py’

previous_chapters.p 100%[===================>]   9.67K  --.-KB/s    in 0s      

2025-10-16 10:54:23 (75.8 MB/s) - ‘previous_chapter

In [ ]:
import torch
from previous_chapters import GPTModel

GPT_CONFIG_124M = {
    "vocab_size": 50257, # GPT-2 BPE 토크나이저의 어휘 크기
    "context_length": 256, # 모델 훈련에 필요한 계산 자원을 절감하기 위해 문맥 길이(context_length)를 256 토큰으로 줄인다.
    "emb_dim": 768, 
    "n_heads": 12, # 12개의 어텐션 헤드
    "n_layers": 12, # 12개의 트랜스포머 블록
    "drop_rate": 0.1, # 드롭아웃 확률 - 요즘에는 드롭아웃을 사용하지 않고 LLM을 훈련하는 경우가 많다.
    "qkv_bias": False, # 최신 LLM은 (초기 GPT 모델과 달리) 쿼리, 키, 값 행렬을 위한 nn.Linear 층에서 편향 벡터를 사용하지 않는다.
}

torch.manual_seed(123) # 재현성을 위해 시드 설정
model = GPTModel(GPT_CONFIG_124M) # 모델 초기화
model.eval() # 평가 모드로 전환

GPTModel(
  (tok_emb): Embedding(50257, 768)
  (pos_emb): Embedding(256, 768)
  (drop_emb): Dropout(p=0.1, inplace=False)
  (trf_blocks): Sequential(
    (0): TransformerBlock(
      (att): MultiHeadAttention(
        (W_query): Linear(in_features=768, out_features=768, bias=False)
        (W_key): Linear(in_features=768, out_features=768, bias=False)
        (W_value): Linear(in_features=768, out_features=768, bias=False)
        (out_proj): Linear(in_features=768, out_features=768, bias=True)
        (dropout): Dropout(p=0.1, inplace=False)
      )
      (ff): FeedForward(
        (layers): Sequential(
          (0): Linear(in_features=768, out_features=3072, bias=True)
          (1): GELU()
          (2): Linear(in_features=3072, out_features=768, bias=True)
        )
      )
      (norm1): LayerNorm()
      (norm2): LayerNorm()
      (drop_shortcut): Dropout(p=0.1, inplace=False)
    )
    (1): TransformerBlock(
      (att): MultiHeadAttention(
        (W_query): Linear(in_features

In [7]:
import tiktoken
from previous_chapters import generate_text_simple

def text_to_token_ids(text, tokenizer):
    encoded = tokenizer.encode(text, allowed_special={'<|endoftext|>'})
    print(f"encoded: {encoded}, length: {len(encoded)}")
    
    encoded_tensor = torch.tensor(encoded).unsqueeze(0) # add batch dimension
    print(f"encoded_tensor: {encoded_tensor}, shape: {encoded_tensor.shape}")
    return encoded_tensor

def token_ids_to_text(token_ids, tokenizer):
    flat = token_ids.squeeze(0) # 배치 차원을 삭제합니다
    return tokenizer.decode(flat.tolist())

start_context = "Every effort moves you"
tokenizer = tiktoken.get_encoding("gpt2")

token_ids = generate_text_simple(
    model=model,
    idx=text_to_token_ids(start_context, tokenizer),
    max_new_tokens=10,
    context_size=GPT_CONFIG_124M["context_length"]
)

print("출력 텍스트:\n", token_ids_to_text(token_ids, tokenizer))

encoded: [6109, 3626, 6100, 345], length: 4
encoded_tensor: tensor([[6109, 3626, 6100,  345]]), shape: torch.Size([1, 4])
출력 텍스트:
 Every effort moves you rentingetic wasnم refres RexMeCHicular stren


4. 텍스트 생성 손실 계산하기
    - 데이터를 로드하는 방법과 generate_text_simple 함수로 텍스트를 생성하는 방법을 간단히 정리한다.   
    - generate_text_simple이 내부적으로 수행하는 작업니다.
    - 모델이 아직 훈련을 하지 않아 타킷 토큰과 생성된 토큰이 거의 일치하지 않는다.

        ![간단 정리](image/05-01-processTxt4.png)
        

In [ ]:
# 두 개의 입력
inputs = torch.tensor([[16833, 3626, 6100],   # ["every effort moves",
                       [40,    1107, 588]])   #  "I really like"]

# 두 개의 타깃
#   - 타킷은 한 위치 앞으로 이동한 입력이다
targets = torch.tensor([[3626, 6100, 345  ],  # [" effort moves you",
                        [1107,  588, 11311]]) #  " really like chocolate"]

In [ ]:
# 입력을 모델에 주입하고 각각 3개의 토큰으로 구성된 로짓 벡터를 계산한다.
with torch.no_grad(): # 평가 모드에서는 그래디언트를 계산하지 않는다.
    logits = model(inputs)
    
probas = torch.softmax(logits, dim=-1) # 로짓을 확률로 변환한다.
# 2: 배치 크기, 3: 시퀀스 길이, 50257: 어휘 크기
print("probas shape:", probas.shape) # (2, 3, 50257)

probas shape: torch.Size([2, 3, 50257])


In [ ]:
# softmax 함수로 로짓을 확률로 변환한 후에 이 확률 점수를 텍스트로 다시 변환
token_ids = torch.argmax(probas, dim=-1, keepdim=True) # 가장 높은 확률을 갖는 토큰을 선택
print("생성된 토큰 IDs:\n", token_ids)

# 
print(f"첫 번째 샘플의 타깃: {token_ids_to_text(targets[0], tokenizer)}")
print(f"두 번째 샘플의 출력: {token_ids_to_text(token_ids[1].flatten(), tokenizer)}")



생성된 토큰 IDs:
 tensor([[[16657],
         [  339],
         [42826]],

        [[49906],
         [29669],
         [41751]]])
첫 번째 샘플의 타깃:  effort moves you
두 번째 샘플의 출력:  pressuring empoweredfaith
